# LA Crime Data Cleaning — Project Scope and Cleaning Principles

This notebook prepares two LAPD crime source datasets—**2010–2019** and **2020–2024**—for downstream exploratory data analysis (EDA). The analytical scope is based on **crime occurrence (`DATE OCC`) from January 1, 2010 through December 31, 2024**.

## Project scope

- `DATE OCC` defines the occurrence period used by the analytical dataset.
- `Date Rptd` is constrained to **December 31, 2024** so the exported snapshot does not contain records reported after the project's reporting cutoff.
- Incident rows are preserved whenever possible. A value is changed only when there is an objective reason to treat it as invalid or unusable for analysis.
- Missing source information is generally preserved rather than replaced with guessed values.

## Cleaning philosophy

The notebook follows a conservative and reproducible pattern:

**inspect → decide → apply the smallest justified change → validate the result**

The goal is not to make every column complete. The goal is to remove clearly invalid values, preserve legitimate source missingness, and prevent silent inconsistencies between the two source periods.

## Main decisions

1. Preserve identifier/code fields in their source-native representation where appropriate. `TIME OCC` remains numeric HHMM data and is validated directly before `hour` and `minute` are derived.
2. Remove exact duplicate rows in **both** source periods and audit duplicate `DR_NO` values rather than arbitrarily selecting among non-identical records.
3. Convert clearly invalid victim ages (`<= 0` and `> 100`) to missing.
4. Replace unsupported victim-sex/descent codes with missing and validate every remaining non-null code against the documented sets.
5. Normalize text formatting only when the transformation changes formatting rather than meaning.
6. Treat zero latitude/longitude placeholders as unavailable geographic values while preserving the incident rows.
7. Preserve source-missing premise and weapon descriptions when the source does not provide enough evidence for a defensible replacement.
8. Validate code/description relationships independently within each source period and again after the two periods are considered together.
9. Validate the final combined dataset before export and verify the exported Parquet file after reloading it.

## Workflow

**Load → inspect → clean → validate each source period → cross-period validation → combine chronologically → engineer temporal features → final validation/inspection → export → reload and verify.**

In [ ]:
# Import the numerical and tabular libraries used throughout the cleaning notebook.

import numpy as np
import pandas as pd

In [ ]:
# Load the two raw LAPD source periods separately so each file can be inspected and cleaned before integration.

df_2010_2019 = pd.read_csv("../data/raw/Crime_Data_from_2010_to_2019.csv",low_memory=False)

df_2020_2024 = pd.read_csv("../data/raw/Crime_Data_from_2020_to_2024.csv",low_memory=False,)

print("2010-2019 dataset loaded successfully.")
print("2020-2024 dataset loaded successfully.")

In [ ]:
# Set display options used for exploratory inspection; these settings do not modify the underlying data.

# Configure pandas to display all core features without truncation
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', None)

In [ ]:
# Inspect representative raw records from each source period to understand the initial field values and missingness.

print("2010-2019 — first 5 records:")
display(df_2010_2019.head())

print("\n2020-2024 — first 5 records:")
display(df_2020_2024.head())

In [ ]:
# Inspect the raw column names to identify schema differences between the two source files.

print("2010-2019 columns:")
print(df_2010_2019.columns.tolist())

print("\n" + "=" * 80 + "\n")

print("2020-2024 columns:")
print(df_2020_2024.columns.tolist())

In [ ]:
# Strip accidental leading/trailing whitespace from column names so both periods share one consistent schema.
# As we saw in the previous cell, `AREA` in the 2010_2019 record has a trailing space.
# Will apply strip to both to make sure.
# Standardize column names so both source datasets use the same schema.

df_2010_2019.columns = df_2010_2019.columns.str.strip()
df_2020_2024.columns = df_2020_2024.columns.str.strip()

In [ ]:
# Confirm the starting row and column counts for both source periods before any cleaning removes records or fields.

print("2010-2019 shape:", df_2010_2019.shape)
print("2020-2024 shape:", df_2020_2024.shape)

In [ ]:
# Review source dtypes so later parsing and cleaning decisions are based on the actual input representation.

print("2010-2019 data types:")
display(df_2010_2019.dtypes)

print("\n" + "=" * 80 + "\n")

print("2020-2024 data types:")
display(df_2020_2024.dtypes)

In [ ]:
# Quantify missingness field-by-field in each source period before deciding which missing values require investigation.

missing_2010_2019 = pd.DataFrame({
    "missing_count": df_2010_2019.isna().sum(),
    "missing_percent": df_2010_2019.isna().mean() * 100
}).sort_values("missing_count", ascending=False)

missing_2020_2024 = pd.DataFrame({
    "missing_count": df_2020_2024.isna().sum(),
    "missing_percent": df_2020_2024.isna().mean() * 100
}).sort_values("missing_count", ascending=False)

print("2010-2019 missing values:")
display(missing_2010_2019)

print("\n2020-2024 missing values:")
display(missing_2020_2024)

### Columns excluded from the analytical dataset

The raw source files contain 28 columns. The cleaned EDA dataset intentionally retains a smaller analytical schema.

The following seven fields are excluded:

- `Crm Cd 1–4`: auxiliary crime-code fields that are not required for the project's primary crime classification.
- `Mocodes`: detailed modifier/offense-context codes that are outside the current EDA scope.
- `Cross Street`: a highly sparse location-detail field; the main `LOCATION` field and geographic coordinates are retained instead.
- `Part 1-2`: a broader crime classification field that is not needed once the primary crime code and description are retained.

Dropping these fields is a **scope decision**, not an assertion that the source columns are incorrect. The original raw CSV files remain the source of record.

In [ ]:
# Remove only the explicitly out-of-scope source columns; retain the fields required for the planned EDA.

columns_to_drop = [
    "Crm Cd 1",
    "Crm Cd 2",
    "Crm Cd 3",
    "Crm Cd 4",
    "Mocodes",
    "Cross Street",
    "Part 1-2"
]
df_2010_2019 = df_2010_2019.drop(columns=columns_to_drop)
df_2020_2024 = df_2020_2024.drop(columns=columns_to_drop)

print("2010-2019 shape:", df_2010_2019.shape)
print("2020-2024 shape:", df_2020_2024.shape)

print(
    "Dropped columns still present:",
    set(columns_to_drop) &
    set(df_2010_2019.columns) &
    set(df_2020_2024.columns)
)

In [ ]:
# Recheck missingness after schema reduction so the retained analytical columns can be assessed consistently.

missing_comparison = pd.DataFrame({
    "2010_2019_missing": df_2010_2019.isna().sum(),
    "2010_2019_percent": df_2010_2019.isna().mean() * 100,
    "2020_2024_missing": df_2020_2024.isna().sum(),
    "2020_2024_percent": df_2020_2024.isna().mean() * 100
})


display(missing_comparison)

## Identify Exact Duplicate Records

Duplicate handling is performed before the two source periods are combined.

The workflow distinguishes between:

- **Exact duplicate rows**: every field is identical, so one copy can be safely removed.
- **Duplicate `DR_NO` values with non-identical content**: these are not automatically deleted because doing so could discard source information.

The investigation therefore checks both exact row duplication and identifier duplication, then applies the conservative rule that only exact duplicate rows are removed.

In [ ]:
# Count exact duplicate rows in each source period before any duplicate-removal operation is applied.

print("2010-2019 exact duplicate rows:", df_2010_2019.duplicated().sum())
print("2020-2024 exact duplicate rows:", df_2020_2024.duplicated().sum())

In [ ]:
# Audit duplicate incident identifiers in BOTH source periods.
# Exact duplicate rows are inspected separately. Duplicate DR_NO values are
# monitored because the project expects DR_NO to identify one incident record.
for name, df in [("2010-2019", df_2010_2019), ("2020-2024", df_2020_2024)]:
    duplicates = df[df.duplicated("DR_NO", keep=False)]

    print(f"\n{name}")
    print(f"Duplicate rows: {len(duplicates):,}")
    print(f"Duplicate DR_NO values: {duplicates['DR_NO'].nunique():,}")

In [ ]:
# Display a small sample of duplicated DR_NO records from each source period.
# This is an investigation step only; non-identical duplicate identifiers are
# not silently resolved by choosing an arbitrary row.

for name, df in [("2010-2019", df_2010_2019), ("2020-2024", df_2020_2024)]:
    duplicates = df[df.duplicated("DR_NO", keep=False)]

    print(f"\n{name} — sample duplicated DR_NO records")
    display(duplicates.sort_values("DR_NO").head(10))

In [ ]:
# Remove exact duplicate rows from BOTH source periods.
# Important: non-identical rows sharing a DR_NO are not automatically deleted.
# They require investigation because choosing one record could discard real source information.

for name, df in [("2010-2019", df_2010_2019), ("2020-2024", df_2020_2024)]:
    before = len(df)
    df.drop_duplicates(inplace=True)
    
    print(f"{name}: removed {before - len(df):,} exact duplicates")
    assert df["DR_NO"].is_unique, f"Duplicate DR_NO found in {name}"

# Data Inspection Checkpoints for each column

The following sections are intentionally organized as investigation steps rather than immediate transformations.

For each important field, we first examines representative values, missingness, ranges, or mappings. Cleaning decisions are then made from those observations and followed by explicit validation.

This keeps the cleaning process traceable: a transformation is not applied simply because a value looks unusual in isolation.


## Inspect Occurrence Date (`DATE OCC`)

`DATE OCC` is the primary temporal field for the project because the analytical scope is based on **when the incident occurred**.

The workflow:

1. inspects the raw values and their source dtype,
2. parses them using the observed source format,
3. counts values that fail conversion,
4. normalizes the timestamp to midnight because the time component of this field is not used,
5. sorts each source period chronologically, and
6. asserts that each period stays inside its intended occurrence-date bounds.

This prevents malformed dates or records from outside the project period from silently entering the combined dataset.

In [ ]:
# Inspect the observed occurrence-date range before parsing so the source coverage can be compared with the project scope.

print("2010-2019 occurrence dates:")
print(df_2010_2019["DATE OCC"].head(10))
print("\nMinimum-looking value:", df_2010_2019["DATE OCC"].min())
print("Maximum-looking value:", df_2010_2019["DATE OCC"].max())

print("\n" + "=" * 80 + "\n")

print("2020-2024 occurrence dates:")
print(df_2020_2024["DATE OCC"].head(10))
print("\nMinimum-looking value:", df_2020_2024["DATE OCC"].min())
print("Maximum-looking value:", df_2020_2024["DATE OCC"].max())

In [ ]:
# Verify that DATE OCC contains no meaningful time-of-day component before normalizing it.

print("2010-2019 all times are midnight:", df_2010_2019["DATE OCC"].astype(str).str.endswith("12:00:00 AM").all())

print("2020-2024 all times are midnight:",df_2020_2024["DATE OCC"].astype(str).str.endswith("12:00:00 AM").all())

In [ ]:
# Record the raw DATE OCC dtype before conversion; this documents the source representation.

print("\n2010-2019 dtype:", df_2010_2019["DATE OCC"].dtype)
print("2020-2024 dtype:", df_2020_2024["DATE OCC"].dtype)

In [ ]:
# Parse DATE OCC using the explicit source format, convert failures to missing.
# and normalize timestamps to dates, as we already have another feature holding the time `TIME OCC`

for df in [df_2010_2019, df_2020_2024]:
    df["DATE OCC"] = pd.to_datetime(df["DATE OCC"], format="%Y %b %d %I:%M:%S %p", errors="coerce").dt.normalize()

In [ ]:
# Confirm that DATE OCC has become a datetime field after explicit parsing.

print("\n2010-2019 dtype:", df_2010_2019["DATE OCC"].dtype)
print("2020-2024 dtype:", df_2020_2024["DATE OCC"].dtype)

In [ ]:
# Count failed DATE OCC conversions; the project cannot safely use rows with an unavailable occurrence date.

print("2010-2019 invalid DATE OCC:",df_2010_2019["DATE OCC"].isna().sum())

print("2020-2024 invalid DATE OCC:",df_2020_2024["DATE OCC"].isna().sum())

In [ ]:
# Check the resulting minimum and maximum occurrence dates against the intended source-period windows.

print("2010-2019 minimum DATE OCC:",df_2010_2019["DATE OCC"].min())

print("2010-2019 maximum DATE OCC:",df_2010_2019["DATE OCC"].max())

print("\n" + "=" * 80 + "\n")

print("2020-2024 minimum DATE OCC:",df_2020_2024["DATE OCC"].min())

print("2020-2024 maximum DATE OCC:",df_2020_2024["DATE OCC"].max())

In [ ]:
# Sort each cleaned source period chronologically for easier inspection and deterministic downstream ordering.

df_2010_2019 = df_2010_2019.sort_values(by="DATE OCC").reset_index(drop=True)

df_2020_2024 = df_2020_2024.sort_values(by="DATE OCC").reset_index(drop=True)

In [ ]:
# Define and enforce the complete occurrence-date window for the 2010–2019 source period.
# Define and enforce the complete occurrence-date window for the 2020–2024 source period.

START_2010_2019 = pd.Timestamp("2010-01-01")
END_2010_2019 = pd.Timestamp("2019-12-31")

assert df_2010_2019["DATE OCC"].between(START_2010_2019,END_2010_2019).all()

START_2020_2024 = pd.Timestamp("2020-01-01")
END_2020_2024 = pd.Timestamp("2024-12-31")

assert df_2020_2024["DATE OCC"].between(START_2020_2024,END_2020_2024).all()

## Clean Reporting Date (`Date Rptd`)

`Date Rptd` records the reporting date and is handled separately from `DATE OCC`.

The field is parsed with the same explicit source-date format, invalid conversions are checked, and the project-level reporting cutoff is applied afterward.

The cutoff is an **inclusion rule for the exported snapshot**. It does not replace or redefine the occurrence date.


In [ ]:
# Record the raw reporting-date dtypes before conversion for traceability.

print("\n2010-2019 dtype:", df_2010_2019["Date Rptd"].dtype)
print("2020-2024 dtype:", df_2020_2024["Date Rptd"].dtype)

In [ ]:
# Verify that DATE Rptd contains no meaningful time-of-day component before normalizing it.

print("2010-2019 all times are midnight:",df_2010_2019["Date Rptd"].astype(str).str.endswith("12:00:00 AM").all())

print("2020-2024 all times are midnight:",df_2020_2024["Date Rptd"].astype(str).str.endswith("12:00:00 AM").all())

In [ ]:
DATE_FORMAT = "%Y %b %d %I:%M:%S %p"

for df in [df_2010_2019, df_2020_2024]:
    df["Date Rptd"] = pd.to_datetime(df["Date Rptd"],format=DATE_FORMAT,errors="coerce").dt.normalize()

In [ ]:
# Count invalid reporting-date conversions before applying the reporting-date cutoff.

print("2010-2019 invalid Date Rptd:",df_2010_2019["Date Rptd"].isna().sum())

print("2020-2024 invalid Date Rptd:",df_2020_2024["Date Rptd"].isna().sum())

In [ ]:
# Inspect the reporting-date coverage before restricting the final reporting snapshot.

print("2010-2019 minimum Date Rptd:",df_2010_2019["Date Rptd"].min())

print("2010-2019 maximum Date Rptd:",df_2010_2019["Date Rptd"].max())

print("\n" + "=" * 80 + "\n")

print("2020-2024 minimum Date Rptd:",df_2020_2024["Date Rptd"].min())

print("2020-2024 maximum Date Rptd:",df_2020_2024["Date Rptd"].max())

### Apply the Reporting-Date Cutoff

The occurrence window remains **January 1, 2010 through December 31, 2024**.

In addition, the exported snapshot must not contain incidents with `Date Rptd` after **December 31, 2024**. Those records are therefore removed at this stage.

This is deliberately separate from the `DATE OCC` filter: an incident can occur earlier and still be reported later, so occurrence and reporting dates must not be conflated.

In [ ]:
END_DATE = pd.Timestamp("2024-12-31")

for name, df in [("2010-2019", df_2010_2019), ("2020-2024", df_2020_2024)]:
    assert df["Date Rptd"].notna().all(), f"Invalid Date Rptd in {name}"
    print(f"{name} records after cutoff: {(df['Date Rptd'] > END_DATE).sum():,}")

df_2010_2019 = df_2010_2019[df_2010_2019["Date Rptd"] <= END_DATE].copy()
df_2020_2024 = df_2020_2024[df_2020_2024["Date Rptd"] <= END_DATE].copy()

In [ ]:
# After applying the cutoff, no record should have a reporting date beyond 2024-12-31.

for name, df in [("2010-2019", df_2010_2019), ("2020-2024", df_2020_2024)]:
    records_after_cutoff = (df["Date Rptd"] > END_DATE).sum()
    
    print(f"{name} records after cutoff: {records_after_cutoff:,}")
    
    assert records_after_cutoff == 0, f"Reporting-date cutoff failed for {name}"

In [ ]:
# Confirm the maximum reporting date after the cutoff has been applied to each source period.

print("2010-2019 maximum Date Rptd:",df_2010_2019["Date Rptd"].max())

print("2020-2024 maximum Date Rptd:",df_2020_2024["Date Rptd"].max())

In [ ]:
# Verify that both temporal source fields have the expected datetime dtypes after cleaning.

print(df_2010_2019[["DATE OCC", "Date Rptd"]].dtypes)
print("\n")
print(df_2020_2024[["DATE OCC", "Date Rptd"]].dtypes)

In [ ]:
# Inspect the earliest and latest cleaned repoerted dates in each source period before integration.

print(df_2010_2019["Date Rptd"].head())
print(df_2010_2019["Date Rptd"].tail())
print("\n" + "=" * 80 + "\n")
print(df_2020_2024["Date Rptd"].head())
print(df_2020_2024["Date Rptd"].tail())

## Inspect Occurrence Time (`TIME OCC`)

`TIME OCC` is a source field representing time in **HHMM / military-time form**.

The workflow keeps the original field in its numeric source representation and validates it without replacing it with a formatted clock string. The validation checks both:

- the overall `0–2359` range, and
- the minute component (`00–59`).

Later, the validated HHMM value is used to derive separate `hour` and `minute` features for EDA.

In [ ]:
# TIME OCC is stored as numeric HHMM values, including values with fewer than
# four digits. For example, 1 represents 00:01 (1 minute after midnight).
# We validate the numeric representation without changing the original column.
# The HHMM components will be interpreted later when TIME OCC is converted into a time value.

for name, df in [("2010-2019", df_2010_2019), ("2020-2024", df_2020_2024)]:
    num = pd.to_numeric(df["TIME OCC"], errors="coerce")
    
    assert num.between(0, 2359).all(), f"Invalid TIME OCC values in {name}"
    
    print(f"{name} TIME OCC is valid.")

In [ ]:
# Check that the minute component is between 00 and 59.

for name, df in [("2010-2019", df_2010_2019), ("2020-2024", df_2020_2024)]:
    numeric = pd.to_numeric(df["TIME OCC"], errors="coerce")
    invalid = (numeric % 100 >= 60).sum()

    print(f"{name} invalid minute values: {invalid:,}")
    assert invalid == 0, f"Invalid TIME OCC minutes in {name}"

# Victim Age

`Vict Age` contains a mixture of plausible ages and source values that should not be interpreted as literal ages.

The workflow first quantifies and inspects negative ages, zero values, and values above 100. It then applies one transparent rule:

**values `<= 0` or `> 100` are treated as unavailable and converted to missing.**

The incident row itself is preserved. No statistical imputation is performed, because the source does not provide enough evidence for a reliable replacement value.

A pandas nullable integer (`Int64`) type is used so valid ages remain integer-valued while unavailable ages are represented as `<NA>`.


In [ ]:
# Check victim-age values before cleaning.

for name, df in [("2010-2019", df_2010_2019), ("2020-2024", df_2020_2024)]:
    ages = df["Vict Age"]

    print(f"\n{name}")
    print(f"Missing:     {ages.isna().sum():,}")
    print(f"Age <= 0:    {(ages <= 0).sum():,}")
    print(f"Age < 0:     {(ages < 0).sum():,}")
    print(f"Age == 0:    {(ages == 0).sum():,}")
    print(f"Age > 100:   {(ages > 100).sum():,}")
    print("-" * 30)

In [ ]:
# Display all records with ages above 100 before they are converted to missing.

print("2010-2019 ages above 100:")
display(df_2010_2019.loc[df_2010_2019["Vict Age"] > 100])

print("\n2020-2024 ages above 100:")
display(df_2020_2024.loc[df_2020_2024["Vict Age"] > 100])

In [ ]:
# Display all records with less or equal 0 before they are converted to missing.

print("2010-2019 ages less or equal 0:")
display(df_2010_2019.loc[df_2010_2019["Vict Age"] < 0].head(10))

print("\n2020-2024 ages less or equal 0:")
display(df_2020_2024.loc[df_2020_2024["Vict Age"] < 0].head(10))

In [ ]:
# Use nullable integers so valid ages stay as integers and invalid/missing
# values can be represented as <NA>.

for df in [df_2010_2019, df_2020_2024]:
    df["Vict Age"] = df["Vict Age"].astype("Int64")

    invalid_age = (df["Vict Age"] <= 0) | (df["Vict Age"] > 100)
    df.loc[invalid_age, "Vict Age"] = pd.NA

In [ ]:
# Verify the age column after cleaning.

for name, df in [("2010-2019", df_2010_2019), ("2020-2024", df_2020_2024)]:
    ages = df["Vict Age"]

    print(f"\n{name}")
    print(f"Negative ages: { (ages < 0).sum():,}")
    print(f"Age = 0:      { (ages == 0).sum():,}")
    print(f"Age > 100:    { (ages > 100).sum():,}")
    print(f"Missing ages: { ages.isna().sum():,}")
    print("-" * 30)

## Reference Code Sets

Explicit code sets make categorical cleaning reproducible.

The validation strategy has two layers:

1. targeted cleanup of values already identified as unsupported or legacy in the source, and
2. a broader assertion that **every remaining non-null value** belongs to the accepted reference set.

The `Vict Descent` reference set is kept as codes because the cleaning steps validates the source field rather than replacing it with a broader textual demographic label.

### Victim Demographics

The LAPD crime dataset contains coded information about victim sex and descent.
The codes below follow the definitions provided in the LAPD/Los Angeles Open Data
metadata. The original codes are retained in the dataset rather than replacing
them with newly defined categories.

#### Victim Sex

The `Vict Sex` field uses the following codes:

- `F` — Female
- `M` — Male
- `X` — Unknown

#### Victim Descent

The `Vict Descent` field uses the following LAPD-defined codes:

| Code | Meaning |
|------|---------|
| `A` | Other Asian |
| `B` | Black |
| `C` | Chinese |
| `D` | Cambodian |
| `F` | Filipino |
| `G` | Guamanian |
| `H` | Hispanic/Latin/Mexican |
| `I` | American Indian/Alaskan Native |
| `J` | Japanese |
| `K` | Korean |
| `L` | Laotian |
| `O` | Other |
| `P` | Pacific Islander |
| `S` | Samoan |
| `U` | Hawaiian |
| `V` | Vietnamese |
| `W` | White |
| `X` | Unknown |
| `Z` | Asian Indian |

These codes are treated as categorical values and are retained according to the
source definitions. `X` represents an unknown descent, while `O` represents
"Other" as defined by the LAPD metadata. No additional interpretation is applied
to the categories.

## Clean Victim Sex

This section examines the observed `Vict Sex` values, applies the documented cleanup rule, and then validates the result.

The project keeps `F`, `M`, and `X` as the accepted non-null values. The source values `H`, `N`, and `-` are treated as unsupported/legacy values and converted to missing rather than being interpreted or inferred into another category.


In [ ]:
# Profile the observed victim-sex codes separately for each source period before recoding unsupported values.
# dropna=False tells pandas to include missing values (NaN/<NA>) in the count.

for name, df in {
    "2010-2019": df_2010_2019,
    "2020-2024": df_2020_2024}.items():

    print(f"\n{name}")
    print(df["Vict Sex"].value_counts(dropna=False))

### Victim Sex — cleaning rule

The accepted analytical values are `F`, `M`, and `X`.

The source values `H`, `N`, and `-` are replaced with missing values. This is intentionally conservative: the workfolw does not infer a replacement category from other incident attributes.

The following validation checks both the specifically targeted codes and the complete accepted set, so an unexpected non-null value would still be caught.

In [ ]:
# Replace only the explicitly identified unsupported/legacy victim-sex codes; all other values remain unchanged.

invalid_codes = ['H', 'N', '-']

df_2010_2019['Vict Sex'] = df_2010_2019['Vict Sex'].replace(invalid_codes, np.nan)
df_2020_2024['Vict Sex'] = df_2020_2024['Vict Sex'].replace(invalid_codes, np.nan)

In [ ]:
# Re-profile Vict Sex after cleanup so the effect of the targeted replacement is visible.

for name, df in {
    "2010-2019": df_2010_2019,
    "2020-2024": df_2020_2024}.items():

    print(f"\n{name}")
    print(df["Vict Sex"].value_counts(dropna=False))

## Clean Victim Descent

This section treats the source value `-` as an unavailable/legacy placeholder and converts it to missing.

All other non-null values are then checked against the documented descent-code set. The cleaning work does not infer descent from names, locations, or other fields.

In [ ]:
# Profile observed victim-descent codes before applying the documented placeholder cleanup.

print("Vict Descent — 2010-2019")
display(df_2010_2019["Vict Descent"].value_counts(dropna=False))

print("Vict Descent — 2020-2024")
display(df_2020_2024["Vict Descent"].value_counts(dropna=False))

In [ ]:
# Replace the explicit legacy/placeholder descent value with missing while preserving all other source codes.

# Replace the invalid / legacy descent code with a missing value.
df_2010_2019["Vict Descent"] = df_2010_2019["Vict Descent"].replace("-", np.nan)

df_2020_2024["Vict Descent"] = df_2020_2024["Vict Descent"].replace("-", np.nan)

In [ ]:
# Validate Vict Descent codes.

VALID_DESCENT_CODES = {
    "A", "B", "C", "D", "F", "G", "H", "I", "J",
    "K", "L", "O", "P", "S", "U", "V", "W", "X", "Z"
}

for name, df in [("2010-2019", df_2010_2019), ("2020-2024", df_2020_2024)]:
    invalid = (~df["Vict Descent"].dropna().isin(VALID_DESCENT_CODES)).sum()

    print(f"{name} invalid Vict Descent values: {invalid:,}")
    assert invalid == 0, f"Invalid Vict Descent values remain in {name}."

## Clean Incident Location Text

`LOCATION` is a textual source field describing the reported incident location.

The workflow first inspects missingness, repeated spaces, and blank-looking values. It then performs a **formatting-only normalization**:

- collapse repeated whitespace to one space, and
- remove leading/trailing whitespace.

No location category is renamed or inferred, and records are not removed simply because location text is missing.

In [ ]:
# Inspect representative LOCATION values before whitespace normalization to understand the source formatting.

print("2010-2019 LOCATION — first 10 values:")
print(df_2010_2019["LOCATION"].head(10))

print("\n2020-2024 LOCATION — first 10 values:")
print(df_2020_2024["LOCATION"].head(10))

In [ ]:
# Measure missing LOCATION values without removing those incident rows.

print("2010-2019 missing LOCATION:",df_2010_2019["LOCATION"].isna().sum())

print("2020-2024 missing LOCATION:",df_2020_2024["LOCATION"].isna().sum())

In [ ]:
# Count unique LOCATION strings to understand the variety of textual location values before normalization.

print("2010-2019 unique LOCATION values:",df_2010_2019["LOCATION"].nunique())

print("2020-2024 unique LOCATION values:",df_2020_2024["LOCATION"].nunique())

In [ ]:
# Inspect the most frequent LOCATION values to identify common formatting patterns and categories.

print("2010-2019 top 20 LOCATION values:")
display(df_2010_2019["LOCATION"].value_counts().head(10))

print("\n2020-2024 top 20 LOCATION values:")
display(df_2020_2024["LOCATION"].value_counts().head(10))

In [ ]:
# Check LOCATION for extra spaces and blank values.

for name, df in [("2010-2019", df_2010_2019), ("2020-2024", df_2020_2024)]:
    location = df["LOCATION"]

    leading = location.str.startswith(" ").sum()
    trailing = location.str.endswith(" ").sum()
    blank = location.str.strip().eq("").sum()

    print(f"\n{name}")
    print(f"Leading spaces: {leading:,}")
    print(f"Trailing spaces: {trailing:,}")
    print(f"Blank values: {blank:,}")

In [ ]:
# Check for repeated spaces inside LOCATION values.

for name, df in [("2010-2019", df_2010_2019), ("2020-2024", df_2020_2024)]:
    repeated = df["LOCATION"].str.contains(r" {2,}", regex=True).sum()

    print(f"{name} repeated spaces: {repeated:,}")

In [ ]:
# Compare the original LOCATION values with their whitespace-normalized versions.
# The comparison is performed row by row and returns a Boolean Series:
# True  = the value would change after normalization.
# False = the value would remain unchanged.
#
# The Boolean Series is stored in `changed`, so it can be used to identify
# exactly which LOCATION values are affected. When `changed.sum()` is used,
# pandas treats True as 1 and False as 0, giving the total number of values
# that would change.
#
# This comparison does not modify the original LOCATION column.

for name, df in [("2010-2019", df_2010_2019), ("2020-2024", df_2020_2024)]:
    loc = df["LOCATION"]
    normalized = loc.str.replace(r"\s+", " ", regex=True).str.strip()
    changed = loc != normalized

    print(f"\n{name}")
    print(f"Values affected: {changed.sum():,}")

    display(pd.DataFrame({
        "Original": loc[changed].head(10).values,
        "Normalized": normalized[changed].head(10).values
    }))

In [ ]:
# Clean LOCATION formatting without changing the location meaning.
# Replace repeated whitespace with one space and remove surrounding spaces.

for df in [df_2010_2019, df_2020_2024]:
    df["LOCATION"] = (df["LOCATION"].str.replace(r"\s+", " ", regex=True).str.strip())

In [ ]:
# Verify that repeated whitespace has been removed from LOCATION.

for name, df in [("2010-2019", df_2010_2019), ("2020-2024", df_2020_2024)]:
    repeated = df["LOCATION"].str.contains(r"\s{2,}", regex=True).sum()

    print(f"{name} repeated spaces: {repeated:,}")
    assert repeated == 0, f"Repeated whitespace remains in {name}."

In [ ]:
# Display a small post-cleaning LOCATION sample as a visual sanity check.

df_2010_2019["LOCATION"].head(5)

## Inspect Geographic Coordinates

`LAT` and `LON` support future spatial analysis, but the source contains zero-valued placeholders that do not represent a useful incident location.

The notebook therefore inspects missingness, zero values, observed ranges, and the affected records before applying a conservative replacement. Only invalid placeholder components are changed; the incident records themselves remain in the dataset.


In [ ]:
# Inspect sample latitude and longitude values before geographic cleaning.

print("2010-2019")
display(df_2010_2019[["LAT", "LON"]].head())

print("\n" + "=" * 80 + "\n")

print("2020-2024")
display(df_2020_2024[["LAT", "LON"]].head())

In [ ]:
# Check missing latitude and longitude values before geographic cleaning.

for name, df in [("2010-2019", df_2010_2019), ("2020-2024", df_2020_2024)]:
    print(f"\n{name}")
    print(f"Missing LAT: {df['LAT'].isna().sum():,}")
    print(f"Missing LON: {df['LON'].isna().sum():,}")

In [ ]:
# Check latitude and longitude ranges for obviously invalid values.

for name, df in [("2010-2019", df_2010_2019), ("2020-2024", df_2020_2024)]:
    print(f"\n{name}")
    print(f"LAT range: {df['LAT'].min()} to {df['LAT'].max()}")
    print(f"LON range: {df['LON'].min()} to {df['LON'].max()}")

In [ ]:
# Check for zero-valued coordinates before treating them as missing.

for name, df in [("2010-2019", df_2010_2019), ("2020-2024", df_2020_2024)]:
    print(f"\n{name}")
    print(f"LAT = 0: { (df['LAT'] == 0).sum():,}")
    print(f"LON = 0: { (df['LON'] == 0).sum():,}")

In [ ]:
# Inspect records with zero longitude before replacing them.

display(
    df_2010_2019.loc[
        df_2010_2019["LON"] == 0,
        ["DATE OCC", "LAT", "LON", "LOCATION"]
    ]
)

In [ ]:
# Inspect records with zero longitude before replacing them.

display(
    df_2020_2024.loc[
        df_2020_2024["LON"] == 0,
        ["DATE OCC", "LAT", "LON", "LOCATION"]
    ]
)

### Handle zero-coordinate placeholders

The source inspection establishes that the zero coordinates are placeholder values rather than meaningful geographic positions.

The cleaning step converts zero latitude/longitude components to missing values and then immediately validates the legal geographic ranges:

- latitude: `-90` to `90`
- longitude: `-180` to `180`

This keeps records with unavailable geography while preventing placeholder coordinates from being treated as real locations.


In [ ]:
# Replace (0, 0) placeholder coordinates with missing values.
# Keep the incident rows and verify that remaining coordinates are valid.

for name, df in [("2010-2019", df_2010_2019), ("2020-2024", df_2020_2024)]:
    print(f"\n{name}")
    print(f"LAT = 0 before cleanup: {(df['LAT'] == 0).sum():,}")
    print(f"LON = 0 before cleanup: {(df['LON'] == 0).sum():,}")

    df["LAT"] = df["LAT"].replace(0, np.nan)
    df["LON"] = df["LON"].replace(0, np.nan)

    assert df["LAT"].dropna().between(-90, 90).all(), f"Invalid latitude in {name}"
    assert df["LON"].dropna().between(-180, 180).all(), f"Invalid longitude in {name}"

In [ ]:
# Confirm that no exact (0, 0) coordinate placeholders remain.

for name, df in [("2010-2019", df_2010_2019), ("2020-2024", df_2020_2024)]:
    zeros = ((df["LAT"] == 0) & (df["LON"] == 0)).sum()
    print(f"{name} (0, 0) coordinates: {zeros:,}")

## Clean Incident Status

`Status Desc` is the human-readable status field retained for downstream analysis. The coded `Status` field is used during investigation and cleanup, then is excluded from the final analytical schema.

The notebook first inspects common, rare, missing, and unexpected status values. It then removes records containing the explicitly identified unsupported status values and validates that every remaining `Status Desc` belongs to the accepted set.

This avoids silently retaining an unsupported status after the original coded field has been dropped.

### Crime Status Codes Reference

The source `Status` field uses short codes that correspond to investigation/disposition categories. This reference is included to make the investigation cells easier to interpret.

The cleaning itself is based on the observed source values and the validated `Status Desc` field; the notebook does not invent new status categories.

The accepted human-readable descriptions retained for the analytical dataset are:

- `Invest Cont`
- `Adult Other`
- `Adult Arrest`
- `Juv Arrest`
- `Juv Other`

The coded `Status` field is not retained in the final analytical schema.


In [ ]:
# Review Status values, including missing values, before cleaning.

for name, df in [("2010-2019", df_2010_2019), ("2020-2024", df_2020_2024)]:
    print(f"\nStatus — {name}")
    display(df["Status"].value_counts(dropna=False))

In [ ]:
# Inspect rare Status codes before deciding how to handle them.

rare_statuses = ["TH", "13", "19"]

for name, df in [("2010-2019", df_2010_2019), ("2020-2024", df_2020_2024)]:
    print(f"\n{name}")
    
    display(df.loc[df["Status"].isin(rare_statuses),["DR_NO", "Status", "Status Desc"]])

In [ ]:
# Inspect records with missing Status and their Status Desc.

for name, df in [("2010-2019", df_2010_2019), ("2020-2024", df_2020_2024)]:
    print(f"\n{name} — missing Status")

    display(df.loc[df["Status"].isna(), ["DR_NO", "Status", "Status Desc"]])

In [ ]:
# Review Status Desc values, including missing values.

for name, df in [("2010-2019", df_2010_2019), ("2020-2024", df_2020_2024)]:
    print(f"\n{name} — Status Desc")
    display(df["Status Desc"].value_counts(dropna=False))

In [ ]:
# Inspect unusual Status codes together with their descriptions.
# and check if the row has good valid info from other columns.

for name, df in [("2010-2019", df_2010_2019), ("2020-2024", df_2020_2024)]:
    unusual = df["Status"].isin(["TH", "13", "19"])

    display(df.loc[unusual])

In [ ]:
# Treat unsupported Status codes as unknown rather than removing the incident records.
# The affected records are retained because the incident data may still be useful.
# Only the unsupported Status information is treated as missing/unknown.

unknown_status = {"TH", "13", "19"}

for df in [df_2010_2019, df_2020_2024]:
    mask = df["Status"].isin(unknown_status)

    df.loc[mask, "Status"] = pd.NA
    df.loc[mask, "Status Desc"] = pd.NA

In [ ]:
# Verify that unsupported Status codes were replaced with missing values.

for name, df in [("2010-2019", df_2010_2019), ("2020-2024", df_2020_2024)]:
    remaining = df["Status"].isin({"TH", "13", "19"}).sum()

    print(f"{name} unsupported Status codes remaining: {remaining:,}")
    assert remaining == 0, f"Unsupported Status codes remain in {name}."

## Inspect Premise Categories

`Premis Desc` describes the premise/location category associated with an incident.

Because missing descriptions do not make the incident row invalid, the notebook **investigates first and preserves source missingness**. It checks whether `Premis Cd` provides an objectively stable mapping to `Premis Desc`.

Only if the source demonstrates a reliable mapping would a correction be considered. In this notebook, no missing premise description is imputed and no incident row is dropped solely because `Premis Desc` is missing.

In [ ]:
# Profile premise descriptions, including missing values, before deciding whether any source-supported recovery is justified.

print("Premis Desc — 2010-2019")
display(df_2010_2019["Premis Desc"].value_counts(dropna=False))

print("\nPremis Desc — 2020-2024")
display( df_2020_2024["Premis Desc"].value_counts(dropna=False))

In [ ]:
# Quantify missing Premis Desc values separately for each source period before making a cleaning decision.

print("2010-2019 missing premise description:", df_2010_2019["Premis Desc"].isna().sum())
print("2020-2024 missing premise description:", df_2020_2024["Premis Desc"].isna().sum())

In [ ]:
# Inspect representative incidents with missing Premis Desc and check whether Premis Cd is still available.

for period, df in [("2010-2019", df_2010_2019), ("2020-2024", df_2020_2024)]:
    print(f"{period} — records with missing Premis Desc:")
    display(df.loc[df["Premis Desc"].isna(), ["DR_NO", "Premis Cd", "Premis Desc"]].head(10))

In [ ]:
# Check whether missing Premis Desc values can be recovered from Premis Cd.
# Analyze each source dataset separately before integration.

for name, data in [("2010-2019", df_2010_2019), ("2020-2024", df_2020_2024)]:

    # Count distinct descriptions associated with each premise code.
    mapping = data.dropna(subset=["Premis Cd", "Premis Desc"]).groupby("Premis Cd")["Premis Desc"].nunique()

    # Identify codes linked to more than one description.
    inconsistent = mapping[mapping > 1]

    # Count missing descriptions with and without an available premise code.
    missing_with_code = (data["Premis Desc"].isna() & data["Premis Cd"].notna()).sum()
    missing_without_code = (data["Premis Desc"].isna() & data["Premis Cd"].isna()).sum()

    # Display the results for each period.
    print(f"\n{name}")
    print("Codes with multiple descriptions:", len(inconsistent))
    print("Missing descriptions with code:", missing_with_code)
    print("Missing descriptions without code:", missing_without_code)

    # Display inconsistent mappings, if any.
    if not inconsistent.empty:
        display(inconsistent)

# Keep source-missing descriptions as missing to avoid unsupported imputation.
print("\nCleaning decision: retain missing descriptions; do not invent values.")

In [ ]:
# Identify premise codes that already have a known description in each dataset.
for name, data in [("2010–2019", df_2010_2019), ("2020–2024", df_2020_2024)]:

    # Build a mapping from premise codes to their known descriptions.
    mapping = (
        data.dropna(subset=["Premis Cd", "Premis Desc"])
        .drop_duplicates("Premis Cd")
        .set_index("Premis Cd")["Premis Desc"]
    )

    # Find records with missing descriptions but available codes.
    missing = data[data["Premis Desc"].isna() & data["Premis Cd"].notna()]

    # Count records whose codes have a known description.
    recoverable = missing["Premis Cd"].isin(mapping.index).sum()

    print(f"\n{name}")
    print("Missing descriptions with codes:", len(missing))
    print("Potentially recoverable records:", recoverable)
    print("Codes without a known description:", len(missing) - recoverable)

## Validate Crime Codes and Descriptions

`Crm Cd` and `Crm Cd Desc` are source-provided classification fields.

The notebook checks whether each crime code maps to exactly one description within each source period. It also investigates codes that appear in only one period rather than automatically treating period-specific codes as errors.

A separate cross-period validation later checks the combined mapping semantics. This helps detect a code whose meaning might appear internally consistent within each file but differ between the two files.

In [ ]:
# Profile common crime descriptions before testing the stability of Crm Cd-to-description mappings.

print("Crm Cd Desc — 2010-2019")
display(df_2010_2019["Crm Cd Desc"].value_counts(dropna=False).head(20))

print("\nCrm Cd Desc — 2020-2024")
display(df_2020_2024["Crm Cd Desc"].value_counts(dropna=False).head(20))

In [ ]:
# Confirm whether any primary crime-code descriptions are missing before relying on their mappings.

print("2010-2019 missing Crm Cd Desc:",df_2010_2019["Crm Cd Desc"].isna().sum())

print("2020-2024 missing Crm Cd Desc:",df_2020_2024["Crm Cd Desc"].isna().sum())

In [ ]:
# Check whether each crime code has multiple descriptions in each dataset.

for name, df in [("2010-2019", df_2010_2019), ("2020-2024", df_2020_2024)]:
    mapping = df.dropna(subset=["Crm Cd", "Crm Cd Desc"]).groupby("Crm Cd")["Crm Cd Desc"].nunique()
    inconsistent = mapping[mapping > 1]

    print(f"{name} — Inconsistent crime codes: {len(inconsistent)}")

    if not inconsistent.empty:
        display(inconsistent)

In [ ]:
# Check crime-code data type, missing values, and unique codes in each period.

for name, df in [("2010-2019", df_2010_2019), ("2020-2024", df_2020_2024)]:
    print(f"\n{name}")
    print("Data type:", df["Crm Cd"].dtype)
    print("Missing values:", df["Crm Cd"].isna().sum())
    print("Unique codes:", df["Crm Cd"].nunique())

In [ ]:
# Compare crime codes between the two periods.

codes_2010 = set(df_2010_2019["Crm Cd"].dropna())
codes_2020 = set(df_2020_2024["Crm Cd"].dropna())

print("Codes only in 2010-2019:", sorted(codes_2010 - codes_2020))
print("Codes only in 2020-2024:", sorted(codes_2020 - codes_2010))

In [ ]:
# Examine crime codes found in only one period and their descriptions.

crime_codes_2010_2019 = set(df_2010_2019["Crm Cd"].dropna())
crime_codes_2020_2024 = set(df_2020_2024["Crm Cd"].dropna())

crime_codes_only_2010_2019 = sorted(crime_codes_2010_2019 - crime_codes_2020_2024)
crime_codes_only_2020_2024 = sorted(crime_codes_2020_2024 - crime_codes_2010_2019)

for name, codes, data in [
    ("2010–2019", crime_codes_only_2010_2019, df_2010_2019),
    ("2020–2024", crime_codes_only_2020_2024, df_2020_2024)
]:
    print(f"\n{name} — Codes unique to this period")

    display(
        data.loc[data["Crm Cd"].isin(codes), ["Crm Cd", "Crm Cd Desc"]]
        .value_counts()
        .sort_index()
    )

In [ ]:
# Check for extra spaces and blank crime descriptions in each dataset.

for name, df in [("2010-2019", df_2010_2019), ("2020-2024", df_2020_2024)]:
    desc = df["Crm Cd Desc"]

    print(f"\n{name}")
    print("Leading spaces:", desc.str.startswith(" ").sum())
    print("Trailing spaces:", desc.str.endswith(" ").sum())
    print("Blank descriptions:", desc.str.strip().eq("").sum())

## Validate Weapon Codes and Descriptions

Weapon information is substantially missing in the source, so broad imputation is avoided.

The notebook instead:

1. inspects the most common weapon descriptions,
2. measures missing descriptions with and without a weapon code,
3. investigates exceptional code/description combinations,
4. examines period-specific weapon codes, and
5. validates the available code-to-description and description-to-code relationships.

A missing weapon description remains missing unless the source provides objective evidence for a correction.


In [ ]:
# Profile the most common weapon descriptions and the extent of source missingness.

print("Weapon Desc — 2010-2019")
display(df_2010_2019["Weapon Desc"].value_counts(dropna=False).head(10))

print("\nWeapon Desc — 2020-2024")
display(df_2020_2024["Weapon Desc"].value_counts(dropna=False).head(10))

In [ ]:
# Check missing weapon descriptions with and without a weapon code.

for name, df in [("2010-2019", df_2010_2019), ("2020-2024", df_2020_2024)]:
    missing = df["Weapon Desc"].isna()

    print(f"\n{name}")
    print("Missing descriptions:", missing.sum())
    print("Missing descriptions with weapon code:", (missing & df["Weapon Used Cd"].notna()).sum())
    print("Missing descriptions without weapon code:", (missing & df["Weapon Used Cd"].isna()).sum())

In [ ]:
exceptional_weapon = df_2010_2019[
    df_2010_2019["Weapon Desc"].isna() & df_2010_2019["Weapon Used Cd"].notna()
]

display(exceptional_weapon)

In [ ]:
# Inspect records with weapon code 222 without changing the original data.
# Retain the record because it contains valid incident information.
# Missing weapon descriptions alone are not grounds for dropping records.
# Leave Weapon Desc as missing unless its value can be verified.
# It seems the raw still have valid information so won't drop it

weapon_code = pd.to_numeric(df_2010_2019["Weapon Used Cd"], errors="coerce")

display(df_2010_2019.loc[weapon_code.eq(222)])



## Validate Area and Reporting District Fields

`AREA`, `AREA NAME`, and `Rpt Dist No` are operational/geographic source identifiers.

The notebook inspects their distributions and missingness and checks whether:

- one `AREA` code maps to multiple `AREA NAME` values, or
- one `AREA NAME` maps to multiple `AREA` codes.

No arbitrary recoding is applied. The purpose of this section is to verify source consistency before the two periods are integrated.

In [ ]:
# Compare area codes and names across both periods.

for name, df in [("2010-2019", df_2010_2019), ("2020-2024", df_2020_2024)]:
    print(f"\n{name} — AREA")
    print(df["AREA"].value_counts(dropna=False).sort_index())

    print(f"\n{name} — AREA NAME")
    print(df["AREA NAME"].value_counts(dropna=False).sort_index())

In [ ]:
# Check missing area codes and names in each dataset.

for name, df in [("2010-2019", df_2010_2019), ("2020-2024", df_2020_2024)]:
    print(f"\n{name}")
    print("Missing AREA:", df["AREA"].isna().sum())
    print("Missing AREA NAME:", df["AREA NAME"].isna().sum())

In [ ]:
# Check whether each AREA code has only one AREA NAME.

for name, df in [("2010-2019", df_2010_2019), ("2020-2024", df_2020_2024)]:
    mapping = df.groupby("AREA")["AREA NAME"].nunique()
    inconsistent = mapping[mapping > 1]

    print(f"\n{name}")
    print("AREA codes with multiple names:", len(inconsistent))

    if not inconsistent.empty:
        display(inconsistent)

## Cross-Period Mapping Validation

Before combining the two datasets, we need to check that codes and their descriptions are consistent across both periods.

We check both directions:

**Code → Description**: Does each code have the same description?
**Description → Code**: Does each description correspond to the same code?

These checks are performed for crime, weapon, and area fields.

Once the mappings are verified, we can combine the datasets.

In [ ]:
# Temporarily combine both datasets to check mappings across periods.
precombine = pd.concat([df_2010_2019, df_2020_2024], ignore_index=True)

# Check code-to-description and description-to-code consistency.
mappings = [
    ("Crm Cd", "Crm Cd Desc", "Crime"),
    ("Weapon Used Cd", "Weapon Desc", "Weapon"),
    ("AREA", "AREA NAME", "Area")
]

for code, desc, name in mappings:
    data = precombine.dropna(subset=[code, desc])

    # Find codes with multiple descriptions and descriptions with multiple codes.
    bad_codes = data.groupby(code)[desc].nunique()
    bad_descs = data.groupby(desc)[code].nunique()

    bad_codes = bad_codes[bad_codes > 1]
    bad_descs = bad_descs[bad_descs > 1]

    print(f"{name}: inconsistent codes = {len(bad_codes)}, inconsistent descriptions = {len(bad_descs)}")

    # Stop if any mapping is inconsistent.
    assert bad_codes.empty, f"Inconsistent {name} codes found."
    assert bad_descs.empty, f"Inconsistent {name} descriptions found."

print("\nAll cross-period mappings are consistent!")

## Final Validation Before Combining the Datasets

Both cleaned source periods must pass a common structural and integrity contract before concatenation.

The validation covers:

- the exact expected schema,
- duplicate-row and `DR_NO` uniqueness checks,
- occurrence-date boundaries,
- reporting-date cutoff,
- victim-age validity,
- coordinate validity,
- accepted categorical codes, and
- final schema equality between the two source periods.

These are hard assertions: a failed check stops the notebook rather than allowing a questionable dataset to continue downstream.

In [ ]:
# Check whether both datasets have identical columns in the same order.
df_2010_2019.columns.tolist() == df_2020_2024.columns.tolist()

In [ ]:
# Display the standardized column order so the expected analytical schema is visible before assertions run.

print(df_2010_2019.columns.tolist())

In [ ]:
# Validate both datasets before combining them.

expected_cols = df_2010_2019.columns.tolist()

datasets = {
    "2010–2019": df_2010_2019,
    "2020–2024": df_2020_2024
}

for name, df in datasets.items():
    print(f"=== {name} Validation ===")

    # Check columns and duplicate records.
    assert df.columns.tolist() == expected_cols, "Unexpected columns."
    assert not df.duplicated().any(), "Duplicate rows found."
    assert df["DR_NO"].is_unique, "Duplicate incident numbers found."

    # Check occurrence and report dates.
    start = START_2010_2019 if name == "2010–2019" else START_2020_2024
    end = END_2010_2019 if name == "2010–2019" else END_2020_2024

    assert df["DATE OCC"].between(start, end).all(), "Invalid occurrence dates."
    assert df["Date Rptd"].le(END_DATE).all(), "Report dates exceed cutoff."

    # Check victim ages and coordinates.
    assert df["Vict Age"].between(1, 100).all(), "Invalid victim ages."
    assert not ((df["LAT"] == 0) & (df["LON"] == 0)).any(), "(0, 0) coordinates found."
    assert df["LAT"].dropna().between(-90, 90).all(), "Invalid latitudes."
    assert df["LON"].dropna().between(-180, 180).all(), "Invalid longitudes."

    # Check valid categorical values.
    assert df["Vict Sex"].dropna().isin(VALID_SEX_CODES).all(), "Invalid sex codes."
    assert df["Vict Descent"].dropna().isin(VALID_DESCENT_CODES).all(), "Invalid descent codes."

    # Display dataset size and missing values.
    print(f"Rows: {len(df):,}")
    missing = df[expected_cols].isna().sum()
    print(missing[missing > 0].to_string() if missing.any() else "No missing values")
    print("-" * 40)

# Confirm both datasets have matching columns.
assert df_2010_2019.columns.equals(df_2020_2024.columns), "Schemas do not match."

print("All validation checks passed!")

## Combine the Cleaned Datasets

The two source periods have now passed their individual and cross-period validation checks.

They are concatenated row-wise into one analytical dataframe and then sorted by `DATE OCC`, producing a single chronological dataset covering the project occurrence period.

No source-period indicator is added because the planned EDA can recover the relevant time period directly from `DATE OCC`, and the two source schemas have already been aligned.

In [ ]:
# Combine both datasets and sort by occurrence date.
df = pd.concat([df_2010_2019, df_2020_2024], ignore_index=True)
df = df.sort_values("DATE OCC").reset_index(drop=True)

# Display the combined dataset size.
print("Dataset shape:", df.shape)
print("Total rows:", len(df))
print("Total columns:", df.shape[1])

In [ ]:
# Validate the combined dataset before feature engineering.

# Check duplicates and date boundaries.
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate DR_NO:", df["DR_NO"].duplicated().sum())
print("Occurrence date range:", df["DATE OCC"].min(), "to", df["DATE OCC"].max())
print("Report dates before occurrence:", (df["Date Rptd"] < df["DATE OCC"]).sum())

# Summarize remaining missing values.
missing = df.isna().sum()
print("\nMissing values:")
print(missing[missing > 0].to_string() if missing.any() else "None")

# Enforce integration requirements.
assert not df.duplicated().any(), "Duplicate rows found."
assert df["DR_NO"].is_unique, "Duplicate incident numbers found."
assert df["DATE OCC"].between("2010-01-01", "2024-12-31").all(), "Occurrence dates out of range."
assert df["DATE OCC"].is_monotonic_increasing, "Dataset is not chronologically sorted."
assert df["Date Rptd"].le(END_DATE).all(), "Report dates exceed cutoff."

## Feature Engineering: Date and Time

Feature engineering is performed **after source cleaning and validation** so that derived features are built from validated source fields.

The notebook adds:

- `year`, `month`, and `month_name`
- `day_of_week` and `day_name`
- `hour` and `minute` derived from `TIME OCC`
- `reporting_lag_days`, calculated as `Date Rptd - DATE OCC`

The original occurrence/reporting dates remain the source temporal fields. If a negative reporting lag ever appears, only the derived lag is set to missing; the underlying incident is preserved.

In [ ]:
# Create time-based features for analysis.

# Ensure date columns use datetime format.
df["DATE OCC"] = pd.to_datetime(df["DATE OCC"])
df["Date Rptd"] = pd.to_datetime(df["Date Rptd"])

# Extract date features.
df["year"] = df["DATE OCC"].dt.year
df["month"] = df["DATE OCC"].dt.month
df["month_name"] = df["DATE OCC"].dt.month_name()
df["day_of_week"] = df["DATE OCC"].dt.dayofweek
df["day_name"] = df["DATE OCC"].dt.day_name()

# Extract hour and minute from TIME OCC (HHMM format).
time = pd.to_numeric(df["TIME OCC"], errors="coerce")
df["hour"] = (time // 100).astype("Int64")
df["minute"] = (time % 100).astype("Int64")

# Calculate the delay between occurrence and reporting.
df["reporting_lag_days"] = (
    df["Date Rptd"] - df["DATE OCC"]
).dt.days.astype("Int64")

# Replace negative reporting delays with missing values.
negative_lags = df["reporting_lag_days"] < 0
print("Negative reporting lags:", negative_lags.sum())
df.loc[negative_lags, "reporting_lag_days"] = pd.NA

# Preview the new features.
display(df[[
    "DATE OCC", "Date Rptd", "TIME OCC", "year", "month",
    "month_name", "day_of_week", "day_name", "hour", "minute",
    "reporting_lag_days"
]].head())

## Validate Engineered Temporal Features

The derived temporal fields receive their own validation gate after feature engineering.

The assertions verify:

- year remains inside the project period,
- month is `1–12`,
- hour is `0–23`,
- minute is `0–59`, and
- no negative reporting lag remains.

This confirms that the feature-engineering step did not introduce impossible temporal values.


In [ ]:
# Validate the ranges of the engineered temporal features.

assert df["year"].between(2010, 2024).all(), "Invalid year."
assert df["month"].between(1, 12).all(), "Invalid month."
assert df["hour"].between(0, 23).all(), "Invalid hour."
assert df["minute"].between(0, 59).all(), "Invalid minute."
assert df["reporting_lag_days"].dropna().ge(0).all(), "Negative reporting lag found."

print("All temporal feature checks passed!")

## Cleaning Decision Summary

The analytical dataset follows documented, source-based cleaning and validation rules.

The main principle is that **missing values are not automatically treated as invalid data**. Records are retained unless there is a clear reason to remove them. Unsupported values are marked as missing rather than guessed.

| Issue / Field | Action | Reason |
|---|---|---|
| Exact duplicate rows | Check and remove confirmed duplicates | Prevents duplicate records from affecting counts |
| Duplicate `DR_NO` | Check uniqueness and investigate duplicates | Avoids unnecessary data loss |
| `DATE OCC` | Convert to datetime and enforce 2010–2024 limits | Defines the study period |
| `Date Rptd` | Convert to datetime and enforce the December 31, 2024 cutoff | Defines the reporting period |
| `TIME OCC` | Preserve the original HHMM format and validate extracted hours and minutes | Supports time-based analysis |
| Invalid `Vict Age` | Replace ages below 1 or above 100 with missing values, if applied | Avoids using implausible ages |
| Invalid `Vict Sex` / `Vict Descent` | Replace unsupported codes with missing values, if applied | Avoids unsupported recoding |
| Zero coordinates | Replace zero latitude or longitude components with missing values, if applied | Handles possible placeholder values |
| Missing `Premis Desc` / `Weapon Desc` | Preserve missing values when no reliable mapping exists | Prevents fabricated descriptions |
| Code/description relationships | Validate mappings within and across both periods | Detects inconsistent code meanings |
| Negative reporting lag | Replace negative derived values with missing values while retaining the incident | Prevents invalid reporting delays from affecting analysis |

### Export and Round-Trip Verification

The processed analytical dataset is written to Parquet using the validated final dataframe.

Immediately after export, the file is reloaded and checked against the in-memory dataframe for:

- identical shape,
- identical column order,
- unique `DR_NO`,
- chronological `DATE OCC`.

This round-trip check confirms that the exported artifact can be read successfully and still satisfies the key final integrity conditions.


In [ ]:
# Export the final dataset to Parquet and verify it after reloading.

path = "../data/processed/crime_2010_2024.parquet"

df.to_parquet(path, index=False, engine="pyarrow")
df_parquet = pd.read_parquet(path)

# Check dataset size and date range.
print("Shape:", df_parquet.shape)
print("Date range:", df_parquet["DATE OCC"].min(), "to", df_parquet["DATE OCC"].max())

# Preview the data and column types.
display(df_parquet.head())
df_parquet.info()

# Verify that the saved dataset matches the original.
assert df_parquet.shape == df.shape, "Dataset shape mismatch."
assert df_parquet.columns.tolist() == df.columns.tolist(), "Column mismatch."
assert df_parquet["DR_NO"].is_unique, "Duplicate incident numbers found."
assert df_parquet["DATE OCC"].is_monotonic_increasing, "Dates are not sorted."

print("Parquet export and validation successful!")